# 📥 Notebook 12: Preparação de Entrada Operacional (XLSX -> Anonimizado)

Este notebook automatiza a conversão de arquivos brutos de reserva (formato XLSX do sistema) para o formato anonimizado padronizado.

**Padronização Segue o Arquivo:** `reservas_anonimas.csv` (data, id_aluno_anonimo, turma, categoria_reserva, dia_semana)

**Instruções:**
1. Coloque o arquivo `.xlsx` na pasta `../data/predicao/`.
2. Execute este notebook.
3. O arquivo `reservas_consolidadas_predicao.csv` será gerado para o Notebook 13.

# Metodologia Design Science Research (DSR) - NB12

## 1. Problema e Motivação
A necessidade de um processo automatizado e seguro para ingestão de dados brutos (XLSX) que garanta a privacidade dos usuários (LGPD) e a integridade do link entre treino e predição operacional.

## 2. Objetivos da Solução
Desenvolver um artefato de pré-processamento que realize a anonimização determinística (SHA-256 com salt) e a padronização de formatos para o motor de predição.

## 3. Design e Desenvolvimento
Criação de um pipeline de limpeza e transformação que preserva a identidade anonimizada do aluno através de múltiplos períodos letivos, permitindo rastreabilidade estatística sem exposição de dados pessoais.

## 4. Demonstração
Este notebook demonstra a conversão bem-sucedida de planilhas operacionais em DataFrames prontos para inferência, validando o algoritmo de hash compartilhado com o notebook 01a.

## 5. Avaliação
A avaliação aqui foca na corretude técnica da transformação e na garantia de que nenhum dado sensível vaze para os estágios subsequentes de modelagem.


In [1]:
import pandas as pd
import os
import glob
import hashlib
from datetime import datetime

INPUT_FOLDER = '../data/predicao/'
OUTPUT_FILE = os.path.join(INPUT_FOLDER, 'reservas_consolidadas_predicao.csv')

# IMPORTANTE: Usar EXATAMENTE o mesmo algoritmo e salt do NB01a para garantir
# que o mesmo aluno tenha o mesmo id_aluno_anonimo em todos os pipelines.
CHAVE_SEGURANCA = 'predicao_refeicoes_v1'

def anonimizar_id(valor):
    import math
    if valor is None or (isinstance(valor, float) and math.isnan(valor)):
        return None
    return hashlib.sha256((str(valor) + CHAVE_SEGURANCA).encode()).hexdigest()[:12]

print("Bibliotecas carregadas. Hash: SHA-256 com salt (compativel com NB01a).")

Bibliotecas carregadas. Hash: SHA-256 com salt (compativel com NB01a).


In [2]:
try:
    xlsx_files = glob.glob(os.path.join(INPUT_FOLDER, "*.xlsx"))

    if not xlsx_files:
        print("❌ ERRO: Coloque o arquivo .xlsx na pasta 'data/predicao/'.")
    else:
        latest_xlsx = max(xlsx_files, key=os.path.getctime)
        print(f"📂 Processando: {os.path.basename(latest_xlsx)}")

        df_raw = pd.read_excel(latest_xlsx)
        df_raw.columns = [c.strip().upper() for c in df_raw.columns]
        
        df_anon = pd.DataFrame()
        df_anon["data"] = pd.to_datetime(df_raw["DATA"], dayfirst=True)
        
        col_matr = "MATRÍCULA IQ" if "MATRÍCULA IQ" in df_raw.columns else "MATRICULA"
        df_anon["id_aluno_anonimo"] = df_raw[col_matr].apply(anonimizar_id)
        
        df_anon["turma"] = df_raw["TURMA"]
        
        # Categoria de Reserva (REFEIÇÃO) - Limpeza para extrair apenas o prato
        col_ref = "REFEIÇÃO" if "REFEIÇÃO" in df_raw.columns else "REFEICAO"
        def limpar_refeicao(txt):
            txt = str(txt)
            if "(" in txt:
                return txt.split("(")[0].strip()
            return txt.strip()

        df_anon["categoria_reserva"] = df_raw[col_ref].apply(limpar_refeicao)
        
        dias_pt = {0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira", 3: "Quinta-feira", 4: "Sexta-feira", 5: "Sábado", 6: "Domingo"}
        df_anon["dia_semana"] = df_anon["data"].dt.dayofweek.map(dias_pt)
        
        # Garantir ordem das colunas igual ao reservas_anonimas.csv
        cols_final = ["data", "id_aluno_anonimo", "turma", "categoria_reserva", "dia_semana"]
        df_anon = df_anon[cols_final]
        
        df_anon.to_csv(OUTPUT_FILE, index=False)
        print(f"\n✅ SUCESSO! Arquivo anonimizado gerado com {len(df_anon)} registros.")
        display(df_anon.head())

except Exception as e:
    print(f"❌ Erro: {e}")

📂 Processando: reservas_predicao.xlsx

✅ SUCESSO! Arquivo anonimizado gerado com 191 registros.


,data,id_aluno_anonimo,turma,categoria_reserva,dia_semana
0,2025-08-22,4674412bd2fa,1º A - MEC,PRATO TRADICIONAL,Sexta-feira
1,2025-08-22,dce3c331d630,1º A - MEC,PRATO TRADICIONAL,Sexta-feira
2,2025-08-22,ef29dc683090,1º A - MEC,PRATO TRADICIONAL,Sexta-feira
3,2025-08-22,2877a4c33350,1º A - MEC,PRATO TRADICIONAL,Sexta-feira
4,2025-08-22,2022c28006fb,1º A - MEC,PRATO TRADICIONAL,Sexta-feira
